# 05b — Evaluation generation with repetition penalty 1.1 + blind pairwise files (Task 6)

Accelerator: **GPU T4 x2**, Internet **on**. Expected time: ~30–40 min.
**Inputs:** the saved outputs of the SFT (02) and DPO (03) notebooks. Optionally also the LoRA notebooks
(04, 04b): if their merged models are found, the persona generations are redone with the same decoding.

Same as notebook 05 (greedy), but with `configs/generation_rp1.1.yaml` (repetition penalty 1.1, chosen by
the 05a decoding pilot). Outputs go to separate folders so the greedy results are untouched.
- Main run: `configs/generation_rp1.1.yaml` (repetition penalty 1.1, chosen by the 05a pilot), `TAG = "rp1.1"`
  -> `outputs/generations_rp1.1/`, `outputs/eval_rp1.1/`
- Greedy check: `configs/generation.yaml`, `TAG = ""` -> `outputs/generations/`, `outputs/eval/`

Existing generation files are reused only if they were produced with identical settings.

In [ ]:
REPO_URL = "https://github.com/madgod2004/small-model-alignment-starter.git"
REPO_DIR = "/kaggle/working/small-model-alignment-starter"
BRANCH = "main"

In [ ]:
import os
if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only
%cd {REPO_DIR}
!git log --oneline -1

In [ ]:
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors matplotlib "peft>=0.11.0" "trl>=0.12.0" pytest
!python -m pip install -q -e . --no-deps

In [ ]:
!nvidia-smi
!python scripts/check_env.py --expect_gpus 2

## 1. Locate checkpoints and show the decoding settings

In [ ]:
import glob, json, os

def find_model_dir(run, kind="checkpoints/best"):
    """Locate outputs/<run>/<kind>/model.safetensors inside the attached Kaggle inputs."""
    hits = sorted(glob.glob(f"/kaggle/input/**/outputs/{run}/{kind}/model.safetensors", recursive=True))
    return os.path.dirname(hits[0]) if hits else None

MODELS = {name: find_model_dir(name) for name in ["sft", "dpo_hh", "dpo_uf"]}
for name, path in MODELS.items():
    print(f"{name:8} -> {path}")
assert all(MODELS.values()), "attach the SFT (02) and DPO (03) notebook outputs as inputs"

In [ ]:
GEN_CONFIG = "configs/generation_rp1.1.yaml"   # "configs/generation.yaml" for the greedy check
TAG = "rp1.1"                                  # "" for the greedy check
G = "outputs/generations" + (f"_{TAG}" if TAG else "")
E = "outputs/eval" + (f"_{TAG}" if TAG else "")
os.makedirs(G, exist_ok=True); os.makedirs(E, exist_ok=True)
print("generations ->", G, "| pairs ->", E)

LORA = {name: find_model_dir(name, "merged") for name in ["lora_dpo", "lora_dpo_1ep", "lora_dpo_beta05"]}
print("LoRA merged models:", LORA)
!cat {GEN_CONFIG}

## 2. Sanity check: 3 prompts per DPO model with the final decoding settings

In [ ]:
!python scripts/generate.py --model_path {MODELS["dpo_uf"]} --generator sanity --eval_set eval_sets/alpaca_eval_300_seed42.json --output_path outputs/sanity/dpo_uf.json --config {GEN_CONFIG} --limit 3 --print --overwrite 2>&1 | grep -v -i warn
!python scripts/generate.py --model_path {MODELS["dpo_hh"]} --generator sanity --eval_set eval_sets/hh_rlhf_eval_300_seed42.json --output_path outputs/sanity/dpo_hh.json --config {GEN_CONFIG} --limit 3 --print --overwrite 2>&1 | grep -v -i warn
!rm -rf outputs/sanity

## 3. Full generation (jobs spread over both GPUs)

In [ ]:
ALPACA, HH, PERSONA = "eval_sets/alpaca_eval_300_seed42.json", "eval_sets/hh_rlhf_eval_300_seed42.json", "eval_sets/persona_style_eval.json"
jobs = [
    {"model_path": "Qwen/Qwen2.5-0.5B", "generator": "qwen2.5-0.5b-base", "eval_set": ALPACA, "output_path": f"{G}/base_alpaca_eval.json"},
    {"model_path": "Qwen/Qwen2.5-0.5B", "generator": "qwen2.5-0.5b-base", "eval_set": HH, "output_path": f"{G}/base_hh_eval.json"},
    {"model_path": MODELS["sft"], "generator": "qwen2.5-0.5b-sft", "eval_set": ALPACA, "output_path": f"{G}/sft_alpaca_eval.json"},
    {"model_path": MODELS["sft"], "generator": "qwen2.5-0.5b-sft", "eval_set": HH, "output_path": f"{G}/sft_hh_eval.json"},
    {"model_path": MODELS["dpo_uf"], "generator": "qwen2.5-0.5b-dpo-uf", "eval_set": ALPACA, "output_path": f"{G}/dpo_uf_alpaca_eval.json"},
    {"model_path": MODELS["dpo_hh"], "generator": "qwen2.5-0.5b-dpo-hh", "eval_set": HH, "output_path": f"{G}/dpo_hh_hh_eval.json"},
]
if any(LORA.values()):
    jobs.append({"model_path": "HuggingFaceTB/SmolLM2-360M-Instruct", "generator": "smollm2-360m-instruct", "eval_set": PERSONA,
                 "output_path": f"{G}/smollm_base_persona_eval.json", "prompt_format": "chat_template"})
    for name, path in LORA.items():
        if path:
            jobs.append({"model_path": path, "generator": f"smollm2-360m-{name.replace('_', '-')}", "eval_set": PERSONA,
                         "output_path": f"{G}/{name}_persona_eval.json", "prompt_format": "chat_template"})
for job in jobs:
    job["config"] = GEN_CONFIG
json.dump(jobs, open(f"{E}/generation_jobs.json", "w"), indent=1)
!PYTHONUNBUFFERED=1 python scripts/run_generation_jobs.py --jobs {E}/generation_jobs.json

In [ ]:
!python scripts/generation_stats.py {G}/*_eval.json --json_out {E}/generation_stats.json

## 4. Blind pairwise files (DPO vs SFT; LoRA vs base SmolLM2), written before any judging

In [ ]:
!python scripts/build_pairs.py --candidate {G}/dpo_uf_alpaca_eval.json --candidate_name dpo_uf --baseline {G}/sft_alpaca_eval.json --baseline_name sft --output {E}/dpo_uf_vs_sft_alpaca_pairs.jsonl
!python scripts/build_pairs.py --candidate {G}/dpo_hh_hh_eval.json --candidate_name dpo_hh --baseline {G}/sft_hh_eval.json --baseline_name sft --output {E}/dpo_hh_vs_sft_hh_pairs.jsonl
for name in ["lora_dpo", "lora_dpo_1ep", "lora_dpo_beta05"]:
    if os.path.exists(f"{G}/{name}_persona_eval.json"):
        !python scripts/build_pairs.py --candidate {G}/{name}_persona_eval.json --candidate_name {name} --baseline {G}/smollm_base_persona_eval.json --baseline_name smollm_base --output {E}/{name}_vs_smollm_base_persona_pairs.jsonl

## 5. Archive

In [ ]:
!zip -qr /kaggle/working/eval_generation_outputs{"_" + TAG if TAG else ""}.zip {G} {E} outputs/logs/gen_*.log
!ls -lh /kaggle/working/*.zip